# **V. Análisis de variables de tipo fecha**

## Objetivo
Este notebook tiene el fin de analizar la calidad, formato y rango de las variables tipo fecha presentes en el dataset oncológico de GRD (2019-2024), identificando fechas válidas, nulos reales, valores inválidos y patrones temporales para garantizar consistencia en el procesamiento de datos clínicos.

## Proceso
1. **Configuración inicial**: Especificación de rutas de datos y variables de fecha a analizar
2. **Limpieza de fechas**: Identificación y separación de los nulos reales de valores problemáticos
3. **Detección de formatos**: Reconocimiento de múltiples formatos de fecha (ISO, dd-mm-yyyy, yyyymmdd)
4. **Parseo robusto**: Conversión de strings a datetime con múltiples intentos de parsing
5. **Análisis de calidad**: Evaluar validez, rango temporal y distribución de anomalías
6. **Generación de reportes**: Exportar matrices de análisis por año y variable de fecha

### Columnas de la matriz de análisis:

| Columna | Significado |
|---------|-------------|
| **Año** | Año del análisis (2019-2024) |
| **Columna** | Nombre de la variable de fecha analizada |
| **Formato** | Formato detectado (ISO, dd-mm-yyyy, yyyymmdd, desconocido) |
| **Total** | Número total de registros en el archivo |
| **NULL reales** | Valores realmente nulos (NaN de pandas) |
| **% NULL** | Porcentaje de nulos reales |
| **Inválidos** | Strings que no pudieron parsearse como fecha |
| **% Inválidos** | Porcentaje de fechas inválidas |
| **% Válidos** | Porcentaje de fechas válidas (parseadas exitosamente) |
| **Año min/max** | Rango de años en las fechas válidas |
| **Mes min/max** | Rango de meses (1-12) |
| **Día min/max** | Rango de días del mes (1-31) |

### Cómo interpretar:
- **% Válidos cercano a 100%**: Variable de buena calidad
- **% Inválidos > 5%**: Revisar formatos o limpieza de datos
- **Rango de años sospechoso**: Posibles errores de entrada (ej: año 2099, 1899)
- **Mes/Día fuera de rango**: Indicativo de formato no detectado o corrupción

In [10]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================

import pandas as pd
import os
import re
from collections import Counter

# Ruta de datos oncológicos sin procesar
carpeta = "../../Datos/Datos oncológicos (sin procesar)"

# Ruta de resultados para guardar análisis de fechas
resultados_fecha = "../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas"

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]

# Mapeo de equivalencias para estandarizar nombres de columnas inconsistentes
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",       # Corrige corrupción de encoding (BOM UTF-8)
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"     # Renombra a estándar de proyecto
}

# ============================================================================
# UTILIDADES: Funciones de Limpieza y Parseo de Fechas
# ============================================================================

def limpiar_columna_fecha(col):
    """
    - Descripción: Función que limpia valores problemáticos típicos en columnas de fecha.
                   Específicamente, convierte a string, elimina espacios y normaliza representaciones de nulos.
                   Y reserva formato original para detección automática posterior, sin realizar parseo.
    - Entrada: col (pd.Series): Serie con valores de fecha (posiblemente con ruido)
    - Salida: pd.Series: Serie con strings limpios y placeholders de nulos normalizados
    """
    # Convertir a string y remover espacios
    col = col.astype(str).str.strip()
    # Reemplazar valores vacíos comunes por None
    col = col.mask(col.isin([
        "", "nan", "NaN", "None", "NULL",
        "0000-00-00", "00000000"
    ]), None)
    return col # Devuelve columna limpia sin parsear (formato original preservado)

def detectar_formato(ejemplos):
    """
    - Descripción: Función que detecta el formato de fecha de una muestra de valores.
                   Específicamente, utiliza expresiones regulares para identificar patrones comunes.
                   Además, devuelve el formato más frecuente en la muestra. 
    - Entrada: ejemplos (list): Lista de strings con ejemplos de fechas (hasta 50)    
    - Salida: str: Formato detectado ("yyyy-mm-dd", "dd-mm-yyyy", "yyyymmdd") o "desconocido"
    """
    # Lista para almacenar formato de cada ejemplocontador
    formatos = []
    # Iterar sobre ejemplos y clasificar por patrón
    for val in ejemplos:
        if re.match(r"^\d{4}-\d{2}-\d{2}$", val):
            formatos.append("yyyy-mm-dd") # Patrón año-mes-día
        elif re.match(r"^\d{2}-\d{2}-\d{4}$", val):
            formatos.append("dd-mm-yyyy") # Patrón día-mes-año
        elif re.match(r"^\d{8}$", val):
            formatos.append("yyyymmdd") # Patrón año-mes-día sin separadores

    # Devolver formato más frecuente, o "desconocido" si no hay coincidencias
    if formatos:
        return Counter(formatos).most_common(1)[0][0]
    return "desconocido" # Si no se detecta ningún formato conocido retorna desconocido


def parsear_fechas(col):
    """
    - Descripción: Función que realiza parsing robusto de fechas con múltiples intentos de formato.
                   Combina resultados de parseo sin generar warnings, y mantiene NaT para 
                   valores realmente inválidos.
    - Entrada: col (pd.Series): Serie con strings de fecha de formato potencialmente mixto
    - Salida: pd.Series: Series con objetos datetime parsed, NaT para valores inválidos
    """
    # Intento 1: Parseador ISO (yyyy-mm-dd)
    f1 = pd.to_datetime(col, errors="coerce", format="%Y-%m-%d")
    # Identificar cuáles NO son formato ISO (para segundo intento)
    mask_no_iso = ~col.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    # Intento 2: Parseador flexible solo en no-ISO (asume dd-mm-yyyy)
    f2 = pd.to_datetime(col.where(mask_no_iso), errors="coerce", dayfirst=True)
    # Combinar resultados: usar f1 si es válido, sino usar f2
    fechas = f1.fillna(f2)
    return fechas # Devuelve serie con fechas parseadas, NaT para inválidos


def analizar_fecha_tabla(carpeta, archivos, mapa_columnas, columna_fecha):
    """
    - Descripción: Función que genera un análisis completo de características de una variable de fecha.
                   Itera sobre los datasets anuales (2019-2024), limpia valores problemáticos, 
                   detecta formato de fecha automáticamente, realiza parsing robusto, 
                   clasifica nulos reales vs inválidos, calcula métricas de calidad y rangos temporales, 
                   y exporta resultados a CSV.
    - Entradas:
        - carpeta (str): Ruta carpeta datos oncológicos
        - archivos (list): Lista de nombres de archivos CSV (2019-2024)
        - mapa_columnas (dict): Diccionario mapeo de nombres inconsistentes
        - columna_fecha (str): Nombre de variable de fecha a analizar

    - Salida: pd.DataFrame: Tabla con métricas de análisis (1 fila por año)
    """
    # Lista para acumular análisis por año
    resultados = []
    # =====================================================================
    # PROCESAR CADA AÑO
    # =====================================================================
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año_archivo = archivo[-8:-4]  # Extrae año (ej: "2019")
        # Cargar CSV con estandarización de columnas
        df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
        total = len(df)  # Total de registros en archivo
        # Saltar si columna no existe en este año
        if columna_fecha not in df.columns:
            print(f"  Advertencia: {año_archivo}: Columna '{columna_fecha}' no encontrada")
            continue
        # =====================================================================
        # LIMPIEZA: Distinguir nulos reales de strings basura
        # =====================================================================
        col_original = df[columna_fecha]
        # Markdown nulos originales de pandas (NaN)
        mask_null_original = col_original.isna()
        col = col_original.copy().astype("string")
        # Limpiar: remover espacios en no-nulos
        col[~mask_null_original] = col[~mask_null_original].str.strip()
        # Preservar nulos
        col[mask_null_original] = None
        # Normalizar strings basura a None
        col = col.mask(col.isin(["nan", "NaN", "None", "NULL"]), None)
        # =====================================================================
        # DETECCIÓN DE FORMATO
        # =====================================================================
        # Muestrear hasta 50 valores no nulos para análisis
        ejemplos = col.dropna().head(50).tolist()
        # Detectar formato dominante
        formato_detectado = detectar_formato(ejemplos)
        # =====================================================================
        # PARSEO ROBUSTO (sin warnings)
        # =====================================================================
        # Intento 1: Formato ISO
        f1 = pd.to_datetime(col, errors="coerce", format="%Y-%m-%d")
        # Intento 2: En no-ISO, usa dayfirst=True
        mask_no_iso = ~col.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
        f2 = pd.to_datetime(col.where(mask_no_iso), errors="coerce", dayfirst=True)
        # Combinar: f1 primero, luego f2
        fechas = f1.fillna(f2)
        # =====================================================================
        # CLASIFICACIÓN Y CÁLCULO DE MÉTRICAS
        # =====================================================================
        # Identificar nulos reales (vs strings basura no parseables)
        mask_null_real = col.isna()
        # Identificar invalid dates (string existente pero no parseable)
        mask_invalidos = fechas.isna() & col.notna()
        # Contar cada categoría
        n_null_real = mask_null_real.sum()
        n_invalidos = mask_invalidos.sum()
        n_total_nulos = fechas.isna().sum()
        # Calcular porcentajes
        pct_null_real = (n_null_real / total) * 100
        pct_invalidos = (n_invalidos / total) * 100
        pct_validos = 100 - ((n_total_nulos / total) * 100)
        # =====================================================================
        # EXTRAER RANGOS TEMPORALES
        # =====================================================================
        # Extraer componentes de fecha (ignora NaT)
        años = fechas.dt.year.dropna()
        meses = fechas.dt.month.dropna()
        dias = fechas.dt.day.dropna()
        # =====================================================================
        # AGREGAR FILA A RESULTADOS
        # =====================================================================
        resultados.append({
            "Año": int(año_archivo),
            "Columna": columna_fecha,
            "Formato": formato_detectado,
            "Total": total,
            "NULL reales": n_null_real,
            "% NULL": round(pct_null_real, 2),
            "Inválidos": n_invalidos,
            "% Inválidos": round(pct_invalidos, 2),
            "% Válidos": round(pct_validos, 2),
            "Año min": int(años.min()) if len(años) > 0 else None,
            "Año max": int(años.max()) if len(años) > 0 else None,
            "Mes min": int(meses.min()) if len(meses) > 0 else None,
            "Mes max": int(meses.max()) if len(meses) > 0 else None,
            "Día min": int(dias.min()) if len(dias) > 0 else None,
            "Día max": int(dias.max()) if len(dias) > 0 else None,
        })
    # Convertir resultados a DataFrame y ordenar cronológicamente
    df_resultado = pd.DataFrame(resultados).sort_values("Año")
    # =====================================================================
    # GUARDAR RESULTADO EN CSV
    # =====================================================================
    # Crear directorio si no existe
    os.makedirs(resultados_fecha, exist_ok=True)
    # Guardar con encoding UTF-8-SIG (Excel compatible)
    ruta_salida = os.path.join(
        resultados_fecha,
        f"{columna_fecha.lower()}_analisis_fechas.csv"
    )
    df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"✓ Análisis de '{columna_fecha}' completado")
    print(f"  Guardado en: {ruta_salida}")
    return df_resultado

### **Fecha de nacimiento y fecha de ingreso:** Fechas que indican el día exacto en que nació el paciente y el día exacto en que fue admitido en el hospital para iniciar su episodio clínico.
- **Tipo de variables**: Demográfica (FECHA_NACIMIENTO) y Hospitalaria (FECHA_INGRESO).
- **Análisis de nulos e inconsistencias:** Tienen una calidad excelente, con un 0% de nulos en toda la cohorte. Aunque hay ligeras inconsistencias en el formato, pues en 2023 el formato cambia a dd-mm-yyyy, mientras que el resto usa el estándar ISO yyyy-mm-dd. Además, en 2024 hay 1 registro de fecha de ingreso inválido.
- **Veredicto**: Descartar como fechas crudas, pero utilizarlas para generar la variable derivada ``Edad``, ya que los algoritmos basados en árboles (Random Forest, XGBoost) no procesan objetos de tipo datetime de forma nativa.

In [13]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHA_NACIMIENTO")

✓ Análisis de 'FECHA_NACIMIENTO' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fecha_nacimiento_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHA_NACIMIENTO,yyyy-mm-dd,46241,0,0.0,0,0.0,100.0,1901,2019,1,12,1,31
1,2020,FECHA_NACIMIENTO,yyyy-mm-dd,35294,0,0.0,0,0.0,100.0,1917,2020,1,12,1,31
2,2021,FECHA_NACIMIENTO,yyyy-mm-dd,38836,0,0.0,0,0.0,100.0,1911,2021,1,12,1,31
3,2022,FECHA_NACIMIENTO,yyyy-mm-dd,44827,0,0.0,0,0.0,100.0,1919,2022,1,12,1,31
4,2023,FECHA_NACIMIENTO,yyyy-mm-dd,51784,0,0.0,0,0.0,100.0,1922,2023,1,12,1,31
5,2024,FECHA_NACIMIENTO,yyyy-mm-dd,55111,0,0.0,0,0.0,100.0,1919,2024,1,12,1,31


In [14]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHA_INGRESO")

✓ Análisis de 'FECHA_INGRESO' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fecha_ingreso_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHA_INGRESO,yyyy-mm-dd,46241,0,0.0,0,0.0,100.0,2018,2019,1,12,1,31
1,2020,FECHA_INGRESO,yyyy-mm-dd,35294,0,0.0,0,0.0,100.0,2019,2020,1,12,1,31
2,2021,FECHA_INGRESO,yyyy-mm-dd,38836,0,0.0,0,0.0,100.0,2019,2021,1,12,1,31
3,2022,FECHA_INGRESO,yyyy-mm-dd,44827,0,0.0,0,0.0,100.0,2021,2022,1,12,1,31
4,2023,FECHA_INGRESO,dd-mm-yyyy,51784,0,0.0,0,0.0,100.0,2022,2023,1,12,1,31
5,2024,FECHA_INGRESO,yyyy-mm-dd,55111,0,0.0,1,0.0,100.0,2023,2024,1,12,1,31


### **Fechas de traslado (1 al 9):** Fechas exactas en las que el paciente fue movido de un servicio clínico a otro dentro del mismo hospital.
- **Tipo de variables**: Hospitalaria.
- **Análisis de nulos e inconsistencias:** Tienen un déficit de datos masivo. La FECHATRASLADO1 tiene aproximadamente un 81% de nulos, y a partir de la FECHATRASLADO4 los nulos alcanzan el 99%.
- **Veredicto**: Descartar completamente, ya que contienen una gran cantidad de valores nulos (entre 80% y 90%) y el traslado ocurre durante el transcurso de la hospitalización, lo cual puede provocar fuga de datos.

In [15]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO1")

✓ Análisis de 'FECHATRASLADO1' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO1,yyyy-mm-dd,46241,37787,81.72,0,0.0,18.28,2018,2019,1,12,1,31
1,2020,FECHATRASLADO1,yyyy-mm-dd,35294,27494,77.90,0,0.0,22.10,2019,2020,1,12,1,31
2,2021,FECHATRASLADO1,yyyy-mm-dd,38836,31444,80.97,0,0.0,19.03,2019,2021,1,12,1,31
3,2022,FECHATRASLADO1,yyyy-mm-dd,44827,36687,81.84,0,0.0,18.16,2021,2022,1,12,1,31
4,2023,FECHATRASLADO1,dd-mm-yyyy,51784,42360,81.80,0,0.0,18.20,2022,2023,1,12,1,31
5,2024,FECHATRASLADO1,yyyy-mm-dd,55111,44597,80.92,0,0.0,19.08,2022,2024,1,12,1,31


In [16]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO2")

C:\Users\Carloto\AppData\Local\Temp\ipykernel_17204\379966690.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  f2 = pd.to_datetime(col.where(mask_no_iso), errors="coerce", dayfirst=True)


✓ Análisis de 'FECHATRASLADO2' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado2_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO2,yyyy-mm-dd,46241,41263,89.23,2,0.0,10.76,2018,2019,1,12,1,31
1,2020,FECHATRASLADO2,yyyy-mm-dd,35294,31029,87.92,0,0.0,12.08,2019,2020,1,12,1,31
2,2021,FECHATRASLADO2,yyyy-mm-dd,38836,34990,90.10,0,0.0,9.90,2019,2021,1,12,1,31
3,2022,FECHATRASLADO2,yyyy-mm-dd,44827,40534,90.42,0,0.0,9.58,2021,2022,1,12,1,31
4,2023,FECHATRASLADO2,dd-mm-yyyy,51784,46829,90.43,0,0.0,9.57,2022,2023,1,12,1,31
5,2024,FECHATRASLADO2,yyyy-mm-dd,55111,49713,90.21,0,0.0,9.79,2022,2024,1,12,1,31


In [17]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO3")

✓ Análisis de 'FECHATRASLADO3' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado3_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO3,yyyy-mm-dd,46241,44778,96.84,0,0.0,3.16,2018,2019,1,12,1,31
1,2020,FECHATRASLADO3,yyyy-mm-dd,35294,33960,96.22,0,0.0,3.78,2019,2020,1,12,1,31
2,2021,FECHATRASLADO3,yyyy-mm-dd,38836,37687,97.04,0,0.0,2.96,2020,2021,1,12,1,31
3,2022,FECHATRASLADO3,yyyy-mm-dd,44827,43574,97.20,0,0.0,2.80,2021,2022,1,12,1,31
4,2023,FECHATRASLADO3,dd-mm-yyyy,51784,50295,97.12,0,0.0,2.88,2022,2023,1,12,1,31
5,2024,FECHATRASLADO3,yyyy-mm-dd,55111,53410,96.91,0,0.0,3.09,2023,2024,1,12,1,31


In [18]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO4")

✓ Análisis de 'FECHATRASLADO4' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado4_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO4,yyyy-mm-dd,46241,45664,98.75,0,0.0,1.25,2018,2019,1,12,1,31
1,2020,FECHATRASLADO4,yyyy-mm-dd,35294,34765,98.50,0,0.0,1.50,2019,2020,1,12,1,31
2,2021,FECHATRASLADO4,yyyy-mm-dd,38836,38339,98.72,0,0.0,1.28,2020,2021,1,12,1,31
3,2022,FECHATRASLADO4,yyyy-mm-dd,44827,44355,98.95,0,0.0,1.05,2021,2022,1,12,1,31
4,2023,FECHATRASLADO4,dd-mm-yyyy,51784,51189,98.85,0,0.0,1.15,2022,2023,1,12,1,31
5,2024,FECHATRASLADO4,yyyy-mm-dd,55111,54427,98.76,0,0.0,1.24,2023,2024,1,12,1,31


In [19]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO5")

✓ Análisis de 'FECHATRASLADO5' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado5_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO5,yyyy-mm-dd,46241,46034,99.55,0,0.0,0.45,2018,2019,1,12,1,31
1,2020,FECHATRASLADO5,yyyy-mm-dd,35294,35098,99.44,0,0.0,0.56,2019,2020,1,12,1,31
2,2021,FECHATRASLADO5,yyyy-mm-dd,38836,38643,99.50,0,0.0,0.50,2020,2021,1,12,1,31
3,2022,FECHATRASLADO5,yyyy-mm-dd,44827,44640,99.58,0,0.0,0.42,2021,2022,1,12,1,31
4,2023,FECHATRASLADO5,dd-mm-yyyy,51784,51548,99.54,0,0.0,0.46,2022,2023,1,12,1,31
5,2024,FECHATRASLADO5,yyyy-mm-dd,55111,54843,99.51,0,0.0,0.49,2023,2024,1,12,1,31


In [20]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO6")

✓ Análisis de 'FECHATRASLADO6' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado6_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO6,yyyy-mm-dd,46241,46139,99.78,0,0.0,0.22,2018,2019,1,12,1,30
1,2020,FECHATRASLADO6,yyyy-mm-dd,35294,35194,99.72,0,0.0,0.28,2019,2020,1,12,1,31
2,2021,FECHATRASLADO6,yyyy-mm-dd,38836,38743,99.76,0,0.0,0.24,2020,2021,1,12,1,31
3,2022,FECHATRASLADO6,yyyy-mm-dd,44827,44730,99.78,0,0.0,0.22,2021,2022,1,12,1,31
4,2023,FECHATRASLADO6,dd-mm-yyyy,51784,51664,99.77,0,0.0,0.23,2022,2023,1,12,1,31
5,2024,FECHATRASLADO6,yyyy-mm-dd,55111,54981,99.76,0,0.0,0.24,2023,2024,1,12,1,31


In [21]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO7")

✓ Análisis de 'FECHATRASLADO7' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado7_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO7,yyyy-mm-dd,46241,46198,99.91,0,0.0,0.09,2018,2019,1,12,1,31
1,2020,FECHATRASLADO7,yyyy-mm-dd,35294,35241,99.85,0,0.0,0.15,2019,2020,1,12,1,30
2,2021,FECHATRASLADO7,yyyy-mm-dd,38836,38782,99.86,0,0.0,0.14,2020,2021,1,12,1,31
3,2022,FECHATRASLADO7,yyyy-mm-dd,44827,44786,99.91,0,0.0,0.09,2021,2022,1,12,1,31
4,2023,FECHATRASLADO7,dd-mm-yyyy,51784,51716,99.87,0,0.0,0.13,2022,2023,1,12,1,31
5,2024,FECHATRASLADO7,yyyy-mm-dd,55111,55048,99.89,0,0.0,0.11,2023,2024,1,12,1,30


In [22]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO8")

✓ Análisis de 'FECHATRASLADO8' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado8_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO8,yyyy-mm-dd,46241,46223,99.96,0,0.0,0.04,2018,2019,1,12,2,30
1,2020,FECHATRASLADO8,yyyy-mm-dd,35294,35263,99.91,0,0.0,0.09,2019,2020,1,12,2,31
2,2021,FECHATRASLADO8,yyyy-mm-dd,38836,38809,99.93,0,0.0,0.07,2020,2021,1,12,1,30
3,2022,FECHATRASLADO8,yyyy-mm-dd,44827,44804,99.95,0,0.0,0.05,2021,2022,1,12,4,30
4,2023,FECHATRASLADO8,dd-mm-yyyy,51784,51742,99.92,0,0.0,0.08,2022,2023,1,12,1,31
5,2024,FECHATRASLADO8,yyyy-mm-dd,55111,55080,99.94,0,0.0,0.06,2023,2024,1,12,1,28


In [23]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO9")

✓ Análisis de 'FECHATRASLADO9' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado9_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO9,yyyy-mm-dd,46241,46235,99.99,0,0.0,0.01,2018,2019,4,12,12,30
1,2020,FECHATRASLADO9,yyyy-mm-dd,35294,35277,99.95,0,0.0,0.05,2019,2020,1,12,1,29
2,2021,FECHATRASLADO9,yyyy-mm-dd,38836,38818,99.95,0,0.0,0.05,2020,2021,1,12,1,31
3,2022,FECHATRASLADO9,yyyy-mm-dd,44827,44814,99.97,0,0.0,0.03,2021,2022,1,12,1,27
4,2023,FECHATRASLADO9,dd-mm-yyyy,51784,51756,99.95,0,0.0,0.05,2022,2023,1,11,1,31
5,2024,FECHATRASLADO9,yyyy-mm-dd,55111,55095,99.97,0,0.0,0.03,2023,2024,1,12,2,31


### **Fecha de alta:** Fecha en la que el paciente egresó del hospital (ya sea por alta médica, derivación o fallecimiento).
- **Tipo de variable**: Hospitalaria.
- **Análisis de nulos e inconsistencias:** Tiene 0% de nulos, pero también sufre del cambio de formato a dd-mm-yyyy en 2023.
- **Veredicto**: Descartar completamente, ya que la fecha de egreso es en el cierre del episodio, por ende puede ser una consecuencia directa del desenlace (implicando data leakage).

In [24]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAALTA")

✓ Análisis de 'FECHAALTA' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechaalta_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAALTA,yyyy-mm-dd,46241,0,0.0,0,0.0,100.0,2019,2019,1,12,1,31
1,2020,FECHAALTA,yyyy-mm-dd,35294,0,0.0,0,0.0,100.0,2020,2020,1,12,1,31
2,2021,FECHAALTA,yyyy-mm-dd,38836,0,0.0,0,0.0,100.0,2021,2021,1,12,1,31
3,2022,FECHAALTA,yyyy-mm-dd,44827,0,0.0,0,0.0,100.0,2022,2022,1,12,1,31
4,2023,FECHAALTA,dd-mm-yyyy,51784,0,0.0,0,0.0,100.0,2023,2023,1,12,1,31
5,2024,FECHAALTA,yyyy-mm-dd,55111,0,0.0,0,0.0,100.0,2024,2024,1,12,1,31


### **Fecha de procedimiento:** Fecha del primer procedimiento documentado.
- **Tipo de variable**: Hospitalaria.
- **Análisis de nulos e inconsistencias:** 100% de nulos. Literalmente es una columna vacía en toda la base de datos desde 2019 hasta 2024.
- **Veredicto**: Descartar directamente, es una columna con solo datos vacíos.

In [25]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAPROCEDIMIENTO1")

✓ Análisis de 'FECHAPROCEDIMIENTO1' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechaprocedimiento1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAPROCEDIMIENTO1,desconocido,46241,46241,100.0,0,0.0,0.0,None,None,None,None,None,None
1,2020,FECHAPROCEDIMIENTO1,desconocido,35294,35294,100.0,0,0.0,0.0,None,None,None,None,None,None
2,2021,FECHAPROCEDIMIENTO1,desconocido,38836,38836,100.0,0,0.0,0.0,None,None,None,None,None,None
3,2022,FECHAPROCEDIMIENTO1,desconocido,44827,44827,100.0,0,0.0,0.0,None,None,None,None,None,None
4,2023,FECHAPROCEDIMIENTO1,desconocido,51784,51784,100.0,0,0.0,0.0,None,None,None,None,None,None
5,2024,FECHAPROCEDIMIENTO1,desconocido,55111,55111,100.0,0,0.0,0.0,None,None,None,None,None,None


### **Fecha de intervención:** Fecha de la primera intervención o cirugía del paciente.
- **Tipo de variable**: Hospitalaria.
- **Análisis de nulos e inconsistencias:** Tiene entre un 34% y un 42% de valores nulos (pacientes oncológicos que no fueron operados durante ese episodio). Además, tiene casi un 2% de formatos totalmente inválidos en 2024 y contiene outliers e inconsistencias lógicas severas, por ejemplo se detectaron años como "2000" y "2025" en registros que deberían ser de 2020 o 2023.
- **Veredicto**: Descartar, al igual que la fecha de procedimiento, una intervención ocurre después de la admisión del paciente, lo cual puede provocar fuga de datos temporal.

In [26]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAINTERV1")

✓ Análisis de 'FECHAINTERV1' completado
  Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechainterv1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAINTERV1,yyyy-mm-dd,46241,19699,42.60,0,0.00,57.40,2018,2019,1,12,1,31
1,2020,FECHAINTERV1,yyyy-mm-dd,35294,14242,40.35,0,0.00,59.65,2000,2025,1,12,1,31
2,2021,FECHAINTERV1,yyyy-mm-dd,38836,13915,35.83,0,0.00,64.17,2019,2021,1,12,1,31
3,2022,FECHAINTERV1,yyyy-mm-dd,44827,15715,35.06,0,0.00,64.94,2020,2023,1,12,1,31
4,2023,FECHAINTERV1,dd-mm-yyyy,51784,18007,34.77,0,0.00,65.23,2020,2025,1,12,1,31
5,2024,FECHAINTERV1,yyyy-mm-dd,55111,19861,36.04,970,1.76,62.20,2023,2024,1,12,1,31
